In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

# Load the silver table
df = spark.table("predictstockprices.plstocks.silver_stocks_price")

# Filter only active records
df = df.filter(F.col("ACTIVE") == True)

df = df.withColumn("CLOSE", F.col("CLOSE").cast(DoubleType())) \
       .withColumn("OPEN", F.col("OPEN").cast(DoubleType())) \
       .withColumn("HIGH", F.col("HIGH").cast(DoubleType())) \
       .withColumn("LOW", F.col("LOW").cast(DoubleType())) \
       .withColumn("VOL", F.col("VOL").cast(DoubleType()))

# Sort by ticker and date
window_spec = Window.partitionBy("TICKER").orderBy("DATE")

# --- Basic Returns ---
df = df.withColumn("return_1d", F.round((F.col("CLOSE") / F.lag("CLOSE", 1).over(window_spec)) - 1, 2))
df = df.withColumn("return_5d", F.round((F.col("CLOSE") / F.lag("CLOSE", 5).over(window_spec)) - 1, 2))

# --- Simple Moving Averages (SMA) ---
df = df.withColumn("SMA_5", F.round(F.avg("CLOSE").over(window_spec.rowsBetween(-4, 0)), 2))
df = df.withColumn("SMA_10", F.round(F.avg("CLOSE").over(window_spec.rowsBetween(-9, 0)), 2))

# --- RSI (14-day) ---
delta = F.col("CLOSE") - F.lag("CLOSE", 1).over(window_spec)
gain = F.when(delta > 0, delta).otherwise(0)
loss = F.when(delta < 0, -delta).otherwise(0)

avg_gain = F.avg(gain).over(window_spec.rowsBetween(-13, 0))
avg_loss = F.avg(loss).over(window_spec.rowsBetween(-13, 0))

rs = F.when(avg_loss != 0, avg_gain / avg_loss).otherwise(None)
rsi = F.when(rs.isNotNull(), F.round(100 - (100 / (1 + rs)), 2)).otherwise(None)

df = df.withColumn("RSI_14", rsi)

# --- MACD and MACD Signal ---
ema_12 = F.avg("CLOSE").over(window_spec.rowsBetween(-11, 0))
ema_26 = F.avg("CLOSE").over(window_spec.rowsBetween(-25, 0))
macd = F.round(ema_12 - ema_26, 2)

df = df.withColumn("MACD", macd)

macd_window = Window.partitionBy("TICKER").orderBy("DATE").rowsBetween(-8, 0)
df = df.withColumn("MACD_signal", F.round(F.avg("MACD").over(macd_window), 2))

df = df.withColumn("VOL", F.round(F.col("VOL"), 2))

# Select relevant columns
result_df = df.select(
    "DATE", "TICKER", "CLOSE", "OPEN", "HIGH", "LOW", "VOL",
    "return_1d", "return_5d", "SMA_5", "SMA_10", "RSI_14", "MACD", "MACD_signal"
)

# Show or save as gold table
display(result_df)


In [0]:
result_df.write.mode("overwrite").saveAsTable("predictstockprices.plstocks.gold_time_series_features")